# fitcast (Colab edition)

Scrape job postings → predict where the requirements are → predict if you qualify → (optionally) tailor your resume per job.

This is the in-browser version of the [fitcast](https://github.com/) CLI. **No install, no Python setup** — just an Anthropic API key.

- **Time:** ~5 minutes end-to-end
- **Cost:** ~$0.30 per run (10 jobs scored + 3 tailored resumes) on Claude Sonnet 4.6

---

## How to use

1. **Set your Anthropic API key** (cells 1–2)
2. **Upload your resume** (cells 3–4) — markdown is best, but plain text works
3. **Configure the search** (cell 6) — sources, keywords, how many jobs
4. **Run** (cells 8–9) — pre-rank + analyze
5. **Browse + download results** (cell 11)
6. **(Optional) Tailor your resume** for top matches (cell 13)

You can re-run any cell after changing its inputs.

In [ ]:
!pip install anthropic pydantic requests -q
print("✓ Dependencies installed.")

## 1. Anthropic API key

Get one at [console.anthropic.com/settings/keys](https://console.anthropic.com/settings/keys) (free signup; you'll need to add ~$5 of credit).

**Recommended:** click the 🔑 (key) icon in the left sidebar → **Add new secret** → name it `ANTHROPIC_API_KEY` and paste your key. The next cell will pick it up automatically and your key never appears in the notebook.

If you'd rather paste it directly, just run the next cell — it'll prompt with a hidden input.

In [ ]:
import os
from getpass import getpass

api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get("ANTHROPIC_API_KEY")
    if api_key:
        print("✓ Loaded API key from Colab Secrets.")
except Exception:
    pass

if not api_key:
    api_key = getpass("Paste your Anthropic API key (input is hidden): ")

os.environ["ANTHROPIC_API_KEY"] = api_key
assert api_key, "API key not set."
print(f"✓ API key set ({len(api_key)} chars).")

## 2. Your resume

Upload a `.md` or `.txt` file with your resume. Markdown gives Claude the cleanest signal (bullet points, headings, etc.).

If you don't have a markdown resume, run the upload cell anyway and click **Cancel** — the next cell lets you paste raw text instead.

In [ ]:
from google.colab import files

resume = ""
print("Click 'Choose Files' and select your resume.md — or click Cancel to paste in the next cell.")
try:
    uploaded = files.upload()
    if uploaded:
        filename = next(iter(uploaded))
        resume = uploaded[filename].decode("utf-8", errors="replace")
        print(f"\n✓ Loaded {len(resume)} characters from {filename}.")
except Exception as e:
    print(f"Skipped upload ({e}). Use the next cell to paste your resume.")

In [ ]:
# Optional fallback: paste your resume between the triple quotes if you didn't upload above.
# If `resume` already has content from the upload, this cell is a no-op.

if not resume.strip():
    resume = """
# Your Name

Senior Title | your@email.com

## Summary

Two-to-three sentence professional summary describing your background and what you're targeting.

## Skills

- Languages: Python, ...
- Tools: ...

## Experience

### Company — Title (Jan 2022 – Present)

- Bullet describing major project and impact
- Bullet describing scope of ownership

## Education

**Degree**, University (Year)
""".strip()
    print("⚠️  Loaded placeholder resume — replace it before running the pipeline.")
else:
    print(f"✓ Using uploaded resume ({len(resume)} characters).")

## 3. Configure the search

Form fields below. Change values, then run the cell. Defaults are sensible for a generalist tech / health / data search.

In [ ]:
#@title Search configuration { display-mode: "form" }

#@markdown ### Sources

#@markdown Comma-separated Greenhouse slugs. Find them at `boards.greenhouse.io/<slug>`. The script silently skips 404s, so feel free to over-include.
greenhouse_companies = "recursionpharmaceuticals,ginkgobioworks,absci,flatironhealth,freenome,natera,komodohealth,doximity"  #@param {type:"string"}

#@markdown The Muse query filters — comma-separated.
muse_categories = "Data and Analytics,Healthcare,Project Management,Science and Engineering"  #@param {type:"string"}
muse_levels = "Senior Level,Mid Level"  #@param {type:"string"}
muse_locations = "Flexible / Remote"  #@param {type:"string"}

#@markdown ### Filters
keywords = "data,AI,digital,bioinform,computational,regulatory,product manager,business analyst"  #@param {type:"string"}
posted_within_hours = 168 #@param {type:"integer"}

#@markdown ### Pre-rank (cheap relevance filter using Haiku 4.5)
prerank_enabled = True #@param {type:"boolean"}
prerank_threshold = 5 #@param {type:"slider", min:0, max:10, step:1}
prerank_max_candidates = 100 #@param {type:"slider", min:10, max:300, step:10}

#@markdown ### Deep analysis
max_jobs = 10 #@param {type:"slider", min:1, max:30, step:1}
model = "claude-sonnet-4-6" #@param ["claude-sonnet-4-6", "claude-opus-4-7"]

config = {
    "greenhouse": {"companies": [c.strip() for c in greenhouse_companies.split(",") if c.strip()]},
    "muse": {
        "categories": [c.strip() for c in muse_categories.split(",") if c.strip()],
        "levels": [l.strip() for l in muse_levels.split(",") if l.strip()],
        "locations": [l.strip() for l in muse_locations.split(",") if l.strip()],
        "max_pages": 2,
    },
    "keywords": [k.strip() for k in keywords.split(",") if k.strip()],
    "max_jobs": max_jobs,
    "posted_within_hours": posted_within_hours if posted_within_hours > 0 else None,
    "prerank": {
        "enabled": prerank_enabled,
        "threshold": prerank_threshold,
        "max_candidates": prerank_max_candidates,
    },
}

print(f"✓ Config ready. Will deep-analyze up to {max_jobs} jobs using {model}.")

## 4. Pipeline code

Run the cell below — it loads all the scraping and analysis functions. You don't need to read or modify it unless you want to.

In [ ]:
import json
import random
import re
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timedelta, timezone
from html.parser import HTMLParser
from typing import Literal, Optional

import anthropic
import requests
from pydantic import BaseModel, Field, ValidationError

PRERANK_MODEL = "claude-haiku-4-5"
GREENHOUSE_BASE = "https://boards-api.greenhouse.io/v1/boards"
MUSE_BASE = "https://www.themuse.com/api/public/jobs"

# --- Pydantic models ---

class RequirementsSection(BaseModel):
    found: bool
    section_heading: Optional[str] = None
    text: str = ""

class RequirementEvidence(BaseModel):
    requirement: str
    met: bool
    confidence: Literal["high", "medium", "low"]
    evidence: str

class QualificationMatch(BaseModel):
    score: int = Field(ge=0, le=100)
    verdict: Literal["qualified", "stretch", "not_qualified"]
    requirements: list
    rationale: str

class ATSAssessment(BaseModel):
    ats_score: int = Field(ge=0, le=100)
    keyword_matches: list
    keyword_gaps: list
    format_warnings: list

class JobAnalysis(BaseModel):
    requirements_section: RequirementsSection
    qualification_match: QualificationMatch
    ats_assessment: ATSAssessment

ANALYSIS_SCHEMA = {
    "type": "object",
    "properties": {
        "requirements_section": {
            "type": "object",
            "properties": {
                "found": {"type": "boolean"},
                "section_heading": {"type": ["string", "null"]},
                "text": {"type": "string"},
            },
            "required": ["found", "section_heading", "text"],
            "additionalProperties": False,
        },
        "qualification_match": {
            "type": "object",
            "properties": {
                "score": {"type": "integer"},
                "verdict": {"type": "string", "enum": ["qualified", "stretch", "not_qualified"]},
                "requirements": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "requirement": {"type": "string"},
                            "met": {"type": "boolean"},
                            "confidence": {"type": "string", "enum": ["high", "medium", "low"]},
                            "evidence": {"type": "string"},
                        },
                        "required": ["requirement", "met", "confidence", "evidence"],
                        "additionalProperties": False,
                    },
                },
                "rationale": {"type": "string"},
            },
            "required": ["score", "verdict", "requirements", "rationale"],
            "additionalProperties": False,
        },
        "ats_assessment": {
            "type": "object",
            "properties": {
                "ats_score": {"type": "integer"},
                "keyword_matches": {"type": "array", "items": {"type": "string"}},
                "keyword_gaps": {"type": "array", "items": {"type": "string"}},
                "format_warnings": {"type": "array", "items": {"type": "string"}},
            },
            "required": ["ats_score", "keyword_matches", "keyword_gaps", "format_warnings"],
            "additionalProperties": False,
        },
    },
    "required": ["requirements_section", "qualification_match", "ats_assessment"],
    "additionalProperties": False,
}

# --- HTML → text ---

class _HTMLStripper(HTMLParser):
    def __init__(self):
        super().__init__()
        self.parts = []
    def handle_data(self, data):
        self.parts.append(data)
    def handle_starttag(self, tag, attrs):
        if tag in ("p", "br", "div"): self.parts.append("\n")
        elif tag == "li": self.parts.append("\n- ")
        elif tag in ("h1", "h2", "h3", "h4"): self.parts.append("\n\n")
    def handle_endtag(self, tag):
        if tag in ("p", "div", "h1", "h2", "h3", "h4"): self.parts.append("\n")

def html_to_text(html):
    p = _HTMLStripper()
    p.feed(html or "")
    text = "".join(p.parts)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()

def _parse_iso(ts):
    if not ts: return None
    try:
        if ts.endswith("Z"): ts = ts[:-1] + "+00:00"
        dt = datetime.fromisoformat(ts)
        if dt.tzinfo is None: dt = dt.replace(tzinfo=timezone.utc)
        return dt
    except (ValueError, TypeError): return None

# --- Scrapers ---

def fetch_greenhouse_jobs(slug):
    url = f"{GREENHOUSE_BASE}/{slug}/jobs"
    try:
        r = requests.get(url, params={"content": "true"}, timeout=20)
        r.raise_for_status()
    except requests.RequestException as exc:
        st = getattr(getattr(exc, "response", None), "status_code", None)
        if st != 404:
            print(f"  ! greenhouse/{slug}: {exc}")
        return []
    return [{
        "source": "greenhouse",
        "id": str(j.get("id", "")),
        "title": j.get("title", ""),
        "company": slug,
        "location": (j.get("location") or {}).get("name", ""),
        "url": j.get("absolute_url", ""),
        "content_html": j.get("content", "") or "",
        "posted_at": _parse_iso(j.get("updated_at")),
    } for j in r.json().get("jobs", [])]

def fetch_muse_jobs(categories, levels, locations, max_pages=2):
    jobs = []
    for page in range(max_pages):
        params = [("page", str(page)), ("descending", "true")]
        for c in categories: params.append(("category", c))
        for l in levels: params.append(("level", l))
        for loc in locations: params.append(("location", loc))
        try:
            r = requests.get(MUSE_BASE, params=params, timeout=20)
            r.raise_for_status()
        except requests.RequestException as exc:
            print(f"  ! themuse: {exc}")
            break
        data = r.json()
        for it in data.get("results", []):
            comp = (it.get("company") or {}).get("name", "")
            locs = [(l or {}).get("name", "") for l in (it.get("locations") or [])]
            jobs.append({
                "source": "themuse",
                "id": str(it.get("id", "")),
                "title": it.get("name", ""),
                "company": comp,
                "location": ", ".join(filter(None, locs)),
                "url": (it.get("refs") or {}).get("landing_page", ""),
                "content_html": it.get("contents", "") or "",
                "posted_at": _parse_iso(it.get("publication_date")),
            })
        if page + 1 >= data.get("page_count", 0): break
    return jobs

# --- Pre-rank ---

PRERANK_PROMPT = """Resume (summary):
{resume_summary}

Job: {title} at {company}
Posting (start):
{snippet}

On a 0-10 scale, how relevant is this job to this candidate's actual background and skills?
- 0-3: clearly unrelated
- 4-6: tangentially related, might consider
- 7-10: clearly relevant

Respond with ONLY the integer score, nothing else."""

def make_resume_summary(resume, max_chars=1500):
    if len(resume) <= max_chars: return resume
    t = resume[:max_chars]
    lb = t.rfind("\n\n")
    if lb > max_chars // 2: return t[:lb]
    return t

def prerank_score_one(client, resume_summary, job):
    snippet = html_to_text(job.get("content_html", ""))[:800]
    if not snippet: return 0
    prompt = PRERANK_PROMPT.format(
        resume_summary=resume_summary,
        title=job.get("title", ""),
        company=job.get("company", ""),
        snippet=snippet,
    )
    try:
        r = client.messages.create(
            model=PRERANK_MODEL, max_tokens=10,
            messages=[{"role": "user", "content": prompt}],
        )
    except anthropic.APIError:
        return 5
    text = next((b.text for b in r.content if b.type == "text"), "").strip()
    m = re.search(r"\d+", text)
    return max(0, min(10, int(m.group(0)))) if m else 5

def prerank_jobs(client, resume, jobs, threshold, max_candidates, max_workers=10):
    if not jobs: return jobs
    pool = jobs
    if len(pool) > max_candidates:
        random.seed(42)
        pool = random.sample(pool, max_candidates)
        print(f"Pre-ranking pool capped at {max_candidates} (sampled from {len(jobs)})")
    summary = make_resume_summary(resume)
    print(f"Pre-ranking {len(pool)} jobs with {PRERANK_MODEL}...")
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        scores = list(ex.map(lambda j: prerank_score_one(client, summary, j), pool))
    for job, score in zip(pool, scores): job["prerank_score"] = score
    above = [j for j in pool if j["prerank_score"] >= threshold]
    above.sort(key=lambda j: j["prerank_score"], reverse=True)
    print(f"  -> {len(above)} of {len(pool)} above threshold {threshold}")
    return above

# --- Scrape orchestrator ---

def matches_keywords(job, keywords):
    if not keywords: return True
    h = (job["title"] + " " + job["content_html"]).lower()
    return any(kw.lower() in h for kw in keywords)

def scrape_jobs(config, client, resume):
    keywords = config.get("keywords") or []
    max_jobs = int(config.get("max_jobs", 10))
    posted_within_hours = config.get("posted_within_hours")
    all_jobs = []

    gh = ((config.get("greenhouse") or {}).get("companies")) or []
    if gh:
        print(f"Scraping {len(gh)} Greenhouse boards in parallel...")
        with ThreadPoolExecutor(max_workers=10) as ex:
            for jobs in ex.map(fetch_greenhouse_jobs, gh):
                all_jobs.extend(jobs)
        print(f"  greenhouse: {sum(1 for j in all_jobs if j['source']=='greenhouse')} jobs")

    muse = config.get("muse")
    if muse:
        print("Querying The Muse...")
        m_jobs = fetch_muse_jobs(
            categories=muse.get("categories") or [],
            levels=muse.get("levels") or [],
            locations=muse.get("locations") or [],
            max_pages=int(muse.get("max_pages", 2)),
        )
        print(f"  themuse: {len(m_jobs)} jobs")
        all_jobs.extend(m_jobs)

    fetched = len(all_jobs)
    filtered = [j for j in all_jobs if matches_keywords(j, keywords)]

    if posted_within_hours is not None:
        cutoff = datetime.now(timezone.utc) - timedelta(hours=int(posted_within_hours))
        b = len(filtered)
        filtered = [j for j in filtered if j.get("posted_at") and j["posted_at"] >= cutoff]
        print(f"  date filter (last {posted_within_hours}h): {b} -> {len(filtered)}")

    seen = set()
    deduped = []
    for j in filtered:
        u = j.get("url") or ""
        if u and u in seen: continue
        if u: seen.add(u)
        deduped.append(j)

    print(f"Pool: {fetched} fetched -> {len(filtered)} after filters -> {len(deduped)} after dedup")

    pr = config.get("prerank") or {}
    if pr.get("enabled", False) and len(deduped) > max_jobs:
        deduped = prerank_jobs(
            client=client, resume=resume, jobs=deduped,
            threshold=int(pr.get("threshold", 5)),
            max_candidates=int(pr.get("max_candidates", 100)),
        )
        return deduped[:max_jobs]

    random.seed(42)
    random.shuffle(deduped)
    return deduped[:max_jobs]

# --- Analysis ---

SYSTEM_INSTRUCTIONS = """You are a careful job-application analyst. For each job posting:

1. Find the section that lists minimum requirements / qualifications. Common headings: "Requirements", "Minimum Qualifications", "Basic Qualifications", "What You Bring", "Qualifications", "Required Experience". Quote it verbatim in `requirements_section.text`. If no clear section exists, set `found: false`.

2. Decide the verdict honestly — do not inflate:
   - "qualified": meets all hard requirements
   - "stretch": meets most but missing 1-2 specific items
   - "not_qualified": missing degree level, fundamental skill, or substantial experience

3. For EACH requirement you identified in step 1, output an entry in `requirements` containing:
   - `requirement`: short paraphrase of the requirement
   - `met`: true if resume clearly demonstrates it, false otherwise
   - `confidence`: "high" if resume explicitly supports judgment, "medium" if inferring, "low" if ambiguous
   - `evidence`: specific quote/reference from resume, or "not mentioned in resume" / "resume shows X yrs vs N+ required" for unmet ones. This is the most important field.

4. Score qualification 0-100: 90+ qualified, 60-89 stretch, <60 not qualified.

5. Assess ATS keyword alignment (separate from qualification):
   - `keyword_matches`: posting keywords that DO appear in the resume
   - `keyword_gaps`: important posting keywords MISSING from the resume
   - `format_warnings`: resume format issues (tables, complex layouts) — empty if none
   - `ats_score`: 0-100 keyword density alignment

Be terse. Ground claims in what's actually in the resume."""

def analyze_job(client, resume, job, model):
    text_in = html_to_text(job["content_html"])
    if len(text_in) < 100:
        return None, text_in
    user_msg = f"## Job: {job['title']} at {job['company']}\nLocation: {job['location']}\n\n## Posting:\n{text_in}"
    try:
        r = client.messages.create(
            model=model,
            max_tokens=4000,
            system=[
                {"type": "text", "text": SYSTEM_INSTRUCTIONS},
                {"type": "text", "text": f"## Candidate Resume\n\n{resume}", "cache_control": {"type": "ephemeral"}},
            ],
            thinking={"type": "adaptive"},
            output_config={"effort": "medium", "format": {"type": "json_schema", "schema": ANALYSIS_SCHEMA}},
            messages=[{"role": "user", "content": user_msg}],
        )
    except anthropic.APIError as exc:
        print(f"    ! API error: {exc}")
        return None, text_in
    out = next((b.text for b in r.content if b.type == "text"), "")
    if not out:
        return None, text_in
    try:
        return JobAnalysis.model_validate_json(out), text_in
    except (json.JSONDecodeError, ValidationError) as exc:
        print(f"    ! parse error: {exc}")
        return None, text_in

print("✓ Pipeline code loaded.")

## 5. Run the pipeline

This cell does the actual work — scraping, pre-ranking, and analyzing. Takes 1–2 minutes; cost ~$0.20–$0.30 depending on settings.

In [ ]:
client = anthropic.Anthropic()

print("=" * 60)
print("Starting scrape...")
print("=" * 60)

jobs = scrape_jobs(config, client, resume)
print(f"\n✓ Selected {len(jobs)} jobs for deep analysis.\n")

results = []
if not jobs:
    print("No jobs matched. Try broadening keywords or raising posted_within_hours in cell 6.")
else:
    print("=" * 60)
    print("Deep-analyzing jobs...")
    print("=" * 60)
    for i, job in enumerate(jobs, 1):
        prerank = job.get("prerank_score")
        prerank_str = f" (prerank {prerank}/10)" if prerank is not None else ""
        print(f"\n[{i}/{len(jobs)}] [{job.get('source','?')}]{prerank_str} {job['title']} @ {job['company']}")
        analysis, posting_text = analyze_job(client, resume, job, model)
        if analysis is None:
            print("    skipped")
            continue
        posted_at = job.get("posted_at")
        result = {
            "score": analysis.qualification_match.score,
            "verdict": analysis.qualification_match.verdict,
            "ats_score": analysis.ats_assessment.ats_score,
            "title": job["title"],
            "company": job["company"],
            "location": job["location"],
            "url": job["url"],
            "posted_at": posted_at.isoformat() if posted_at else "",
            "prerank_score": prerank,
            "matched": [r.requirement for r in analysis.qualification_match.requirements if r.met],
            "missing": [r.requirement for r in analysis.qualification_match.requirements if not r.met],
            "requirements_evidence": [r.model_dump() for r in analysis.qualification_match.requirements],
            "rationale": analysis.qualification_match.rationale,
            "ats_keyword_matches": analysis.ats_assessment.keyword_matches,
            "ats_keyword_gaps": analysis.ats_assessment.keyword_gaps,
            "ats_format_warnings": analysis.ats_assessment.format_warnings,
            "requirements_text": analysis.requirements_section.text,
            "posting_text": posting_text,
        }
        results.append(result)
        print(f"    -> {result['verdict']} ({result['score']}/100), ATS {result['ats_score']}/100")

    results.sort(key=lambda r: r["score"], reverse=True)
    print(f"\n✓ Done. {len(results)} jobs analyzed.")

## 6. Browse + download results

The table below is sortable in Colab (click column headers). The CSV/JSON files download to your computer automatically.

In [ ]:
import pandas as pd
from google.colab import files

if not results:
    print("No results to display. Re-run the previous cell or broaden your search.")
else:
    df = pd.DataFrame([{
        "score": r["score"],
        "verdict": r["verdict"],
        "ats": r["ats_score"],
        "title": r["title"],
        "company": r["company"],
        "missing (top)": "; ".join(r["missing"][:5]),
    } for r in results])
    print("Top matches by qualification score:\n")
    display(df)

    print("\nApply links (top 5):")
    for r in results[:5]:
        print(f"  {r['score']:>3}/100  {r['title']} @ {r['company']}")
        print(f"          {r['url']}")

    pd.DataFrame(results).to_csv("results.csv", index=False)
    with open("results.json", "w") as f:
        json.dump(results, f, indent=2, default=str)

    print("\n📥 Downloading results.csv and results.json...")
    files.download("results.csv")
    files.download("results.json")

## 7. (Optional) Tailor your resume for top matches

Pick how many top-ranked jobs to tailor for. Each tailored resume costs ~$0.05.

The model is explicitly told **NOT** to invent skills, inflate experience, or fabricate credentials. Tailoring is *emphasis and vocabulary*, not fiction.

In [ ]:
#@title Tailor settings
top_n = 3 #@param {type:"slider", min:1, max:10, step:1}
min_score = 60 #@param {type:"slider", min:0, max:100, step:5}

TAILOR_SYSTEM = """You are tailoring a candidate's master resume for a specific job posting.

YOUR JOB: rewrite the resume so it leads with the most relevant experience and uses the posting's exact vocabulary where the candidate genuinely has that experience.

YOU MUST NOT:
- Invent skills, technologies, or experience the candidate doesn't have
- Inflate years of experience or scope of responsibility
- Add credentials, degrees, or certifications they don't have
- Change factual details (dates, employers, titles, locations, degree fields)
- Fabricate metrics or achievements

YOU MAY:
- Reorder bullets within each role to lead with the most relevant
- Rephrase bullets to mirror the posting's vocabulary, ONLY when the candidate's actual experience matches
- Drop bullets that are clearly irrelevant (don't drop entire roles)
- Tighten the professional summary to focus on this role's needs
- Surface quantified achievements that align with the posting
- Reorder the skills section to lead with what the posting mentions

OUTPUT:
1. The full tailored resume in clean Markdown — keep it the same length as the original
2. A `## Changes Summary` section listing each change and the line in the original that justifies it

If the candidate isn't qualified at all and tailoring would require inventing experience, say so explicitly in the Changes Summary instead of forcing it."""

def tailor_one(client, resume, result, model):
    posting = result.get("posting_text") or result.get("requirements_text", "")
    if not posting: return ""
    user_msg = f"""## Job Posting

Title: {result.get('title', '')}
Company: {result.get('company', '')}
Location: {result.get('location', 'N/A')}
URL: {result.get('url', '')}

{posting}

## Master Resume

{resume}

---

Now produce the tailored resume in Markdown, followed by a `## Changes Summary` section."""
    r = client.messages.create(
        model=model, max_tokens=8000, system=TAILOR_SYSTEM,
        thinking={"type": "adaptive"}, output_config={"effort": "medium"},
        messages=[{"role": "user", "content": user_msg}],
    )
    return next((b.text for b in r.content if b.type == "text"), "")

def slugify(s):
    s = re.sub(r"[^\w\s-]", "", s).strip().lower()
    return re.sub(r"[\s_-]+", "-", s)[:50]

import os
targets = [r for r in results if r["score"] >= min_score][:top_n]

if not targets:
    print(f"No results with score >= {min_score}. Try lowering min_score or rerunning the analysis.")
else:
    os.makedirs("tailored", exist_ok=True)
    print(f"Tailoring {len(targets)} resume(s)...\n")
    for i, r in enumerate(targets, 1):
        print(f"[{i}/{len(targets)}] {r['title']} @ {r['company']} (score {r['score']})")
        md = tailor_one(client, resume, r, model)
        if not md:
            print("    ! empty response")
            continue
        fn = f"tailored/{slugify(r['company'])}_{slugify(r['title'])}.md"
        header = (
            f"<!--\n"
            f"Tailored resume for: {r['title']} @ {r['company']}\n"
            f"Job URL: {r.get('url', '')}\n"
            f"Generated: {datetime.now(timezone.utc).isoformat()}\n"
            f"Qualification score: {r['score']}/100  ATS score: {r['ats_score']}/100\n"
            f"-->\n\n"
        )
        with open(fn, "w") as f:
            f.write(header + md)
        print(f"    -> {fn}")
        files.download(fn)
    print(f"\n✓ {len(targets)} tailored resume(s) downloaded.")

## 🎉 All done

If this was useful, the source repo (with CLI version, application tracking, and SimplifyJobs bootstrap) is at https://github.com/aa9gj/fitcast.

### Re-running with different settings

- Change form values in cell 6 (search) or cell 13 (tailor)
- Re-run that cell + everything below it (or **Runtime → Run after**)

### Converting tailored markdown to DOCX or PDF for ATS upload

Download the `.md` file, then on your local machine:

```bash
pandoc tailored/foo.md -o foo.docx
```

(Most ATS systems prefer DOCX or PDF over Markdown.)